In [ ]:
#importera
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#läs in fil och omvandla till csv
df_stroop = pd.read_excel("Raw_Stroop.xlsx")
df_stroop.to_csv("Raw_Stroop.csv", index=False )

# Info om dataset:

In [ ]:
df_stroop.head()

In [ ]:
df_stroop.info()

In [ ]:
print(f"---SHAPE---\n{df_stroop.shape}")
print(f"---SUBJECT ID---\n{df_stroop["subjectid"].nunique()}")
print(f"---VALUES.congruence---\n{df_stroop["values.congruency"].value_counts()}")
print(f"---CORRECT---\n{df_stroop["correct"].value_counts()}")

In [ ]:
#Visa sidan "Variables description" i excel filen

desc_stroop = pd.read_excel("Raw_Stroop.xlsx", sheet_name = "Variables Description")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
desc_stroop

In [ ]:
#finns NaNs?
df_stroop.isna().sum()

In [ ]:
df_stroop["computer.platform"].describe()

In [ ]:
df_stroop["computer.platform"].unique()

In [ ]:
computer_platform = df_stroop.groupby("computer.platform")['subjectid'].nunique()
computer_platform

In [ ]:
df_stroop["values.congruency"].unique()

In [ ]:
df_stroop.groupby(["subjectid", "date"]).size().value_counts()

In [ ]:
#definiera session
sessions = df_stroop.groupby(["subjectid", "date", "time"]).size().value_counts()
sessions

In [ ]:
df_stroop["date"].value_counts()

In [ ]:
#hur många sessioner varje person har i stroop
df_stroop.groupby("subjectid")["date"].nunique().value_counts()

1: online
2: online och lab (L1)
3: online, lab och lab: (L1,L2)

In [ ]:
#lägg till kolumn "session_id"
df_stroop["session_id"] = df_stroop.groupby(["subjectid", "date", "time"]).ngroup()

In [ ]:
df_stroop.groupby("session_id").size().value_counts()
df_stroop["session_id"].nunique()

In [ ]:
df_stroop.head()

In [ ]:
df_stroop.head()

## Sortera datum

In [ ]:
#kontrollera hur det ser ut
print(df_stroop["date"].dtype)
print(df_stroop["date"].astype(str).str.len().value_counts())
df_stroop[["date", "time"]].drop_duplicates().head(10)

In [ ]:
#skapa kolumn "datetime"
df_stroop["datetime"] = pd.to_datetime(
    df_stroop["date"].astype(str).str.zfill(6) + " " + df_stroop["time"].astype(str), # gör om datumet från tal till text, fyller på med 0 till texten är 6 tecken lång, lägger till klockslag efter mellanslag
    format="%m%d%y %H:%M:%S" #sätter format till månad/dag/år och timma/minut/sekund
)
df_stroop["datetime"].min(), df_stroop["datetime"].max() 

I filen ligger datum som heltal i månad-dag-år, t.ex 111917 för 19 november 2017, dessutom saknas nollor för månader innan oktober, t.ex 12518, ska vara 012518. Därför har vi fyllt på med nollor så alla datum har sex siffror. Sedan slå ihop date och time till en kolumn "datetime", med formatet månad/dag/år och timma/minut/sekund. Detta kontrolleras sedan med min() och max() för att se första och sista datum&tid, vilket stämmer överens med artikeln över när datan samlades in (2017-05-16 till 2019-04-04). Syftet är att kunna sortera sessionerna i tidsordning och särskilja flera sessioner som en person gjort samma dag.

In [ ]:
df_stroop.head()

In [ ]:
#Läs in sammansatt fil
df_comp = pd.read_excel("STROOP_FLANKERS_SART_web_and_lab.xls")

In [ ]:
# 1. Läs in och kontrollera kolumnerna
comp = pd.read_excel("STROOP_FLANKERS_SART_web_and_lab.xls")
print(comp.columns[:12].tolist())
print(comp["FILTER for completed sessions"].value_counts())

# 2. Personer med exakt 2 datum hos er
n_dates = df_stroop.groupby("subjectid")["date"].nunique()
two_dates = set(n_dates[n_dates == 2].index)

# 3. Personer som bara har "online" i sammansatta filen
only_online = set(comp.loc[comp["FILTER for completed sessions"] == "only online", "subjectid"])

# 4. Snittet, alltså de ni letar efter
suspects = sorted(two_dates & only_online)
print(len(suspects), suspects)

In [ ]:
(df_stroop[df_stroop["subjectid"].isin(suspects)]
 [["subjectid", "datetime"]].drop_duplicates().sort_values(["subjectid", "datetime"]))

In [ ]:
# Ta bort 20 personer med två datum
df_stroop = df_stroop[~df_stroop["subjectid"].isin(suspects)]
df_stroop.groupby("subjectid")["date"].nunique().value_counts()


Ovan har vi tagit bort 20 personer, detta var för att de i vår fil hade två datum, medan de i den sammansatta filen enbart hade 1 datum, dvs. att de enbart gjort online testet. Detta upptäckte vi genom att det stod fel i antal i cell 16 jämfört med vad som stod i artikeln. Detta gjorde vi för att inte göra fel när vi ska märka om personerna gjort online eller i labb, då att de har två datum hade kunnat göra så att vi märker personen som online+labb1, medans de bara gjort ett online test